<a href="https://colab.research.google.com/github/carrisian/del-big-data-al-modelo-predictivo/blob/main/Colab%5CAn%C3%A1lisis%20multivariante%20mediante%20LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import ipywidgets as widgets
from IPython.display import display

# 1. Lista de variables disponibles
lista_variables_todas = ['PM10', 'PM2.5', 'NO2', 'Ozono', 'CO', 'SO2', 'Temp', 'Hum',
                         'Viento_Vel', 'Viento_Dir', 'Presion', 'Nubes', 'UV_Real',
                         'UV_Cielo_Despejado', 'Lluvia', 'Radiacion']

# 2. Selector interactivo
sel_variables = widgets.SelectMultiple(
    options=lista_variables_todas,
    value=['PM10', 'NO2', 'Ozono'], # Valor por defecto
    description='Variables:',
    rows=10,
    disabled=False
)
btn_generar = widgets.Button(description='Generar Gráficos', button_style='primary')
out_g = widgets.Output()

display(sel_variables, btn_generar, out_g)

# 3. Función de generación a demanda
def ejecutar_graficos(b):
    out_g.clear_output()
    vars_a_plot = list(sel_variables.value)

    with out_g:
        if not vars_a_plot:
            print("⚠️ Selecciona al menos una variable.")
            return

        estaciones = list(df['Estacion'].unique()) + ['GLOBAL']
        print(f"🚀 Generando {len(estaciones)} gráficos de barras...")

        for est in estaciones:
            subset = df if est == 'GLOBAL' else df[df['Estacion'] == est]
            resumen_mes = subset.groupby('mes')[vars_a_plot].mean().reset_index()
            resumen_melted = resumen_mes.melt(id_vars='mes', var_name='Variable', value_name='Concentracion')

            plt.figure(figsize=(12, 5))
            sns.barplot(data=resumen_melted, x='mes', y='Concentracion', hue='Variable')
            plt.title(f"Media Mensual: {est}")
            plt.ylabel("Unidades reales")
            plt.grid(axis='y', linestyle='--', alpha=0.5)
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.show()
            plt.close()

btn_generar.on_click(ejecutar_graficos)

SelectMultiple(description='Variables:', index=(0, 2, 3), options=('PM10', 'PM2.5', 'NO2', 'Ozono', 'CO', 'SO2…

Button(button_style='primary', description='Generar Gráficos', style=ButtonStyle())

Output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
import tensorflow as tf
from scipy import stats
from google.colab import drive
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display

# 1. Configuración
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
PATH_BASE = "/content/drive/MyDrive/TFM_Profesorado/"
RUTA_OUTPUT = os.path.join(PATH_BASE, "Modelos_IA_Listos")
os.makedirs(RUTA_OUTPUT, exist_ok=True)

# 2. Carga y Preparación
df = pd.read_parquet(f"{PATH_BASE}Murcia_Dataset_Completo_Global_3H.parquet")
df['time'] = pd.to_datetime(df['time'])
df['anio'] = df['time'].dt.year
features = ['PM10', 'PM2.5', 'NO2', 'Ozono', 'CO', 'SO2', 'Temp', 'Hum',
            'Viento_Vel', 'Viento_Dir', 'Presion', 'Nubes', 'UV_Real',
            'UV_Cielo_Despejado', 'Lluvia', 'Radiacion']

# 3. Tendencias OLS (Rigor Estadístico)
print("🚀 Calculando tendencias OLS...")
tasas_anuales, tasas_mensuales = {}, {}
estaciones = list(df['Estacion'].unique()) + ['GLOBAL']

for est in estaciones:
    sub = df if est == 'GLOBAL' else df[df['Estacion'] == est]
    m_anio = sub.groupby('anio')['NO2'].mean().reset_index()
    slope, _, _, _, _ = stats.linregress(m_anio['anio'], m_anio['NO2'])
    tasas_anuales[est] = slope
    for mes in range(1, 13):
        m = sub[sub['time'].dt.month == mes].groupby('anio')['NO2'].mean().reset_index()
        tasas_mensuales[(est, mes)] = stats.linregress(m['anio'], m['NO2'])[0] if len(m) > 2 else 0

# 4. Entrenamiento Masivo (Sin optimizaciones: 15 épocas, Dropout, LSTM(64))
def create_sequences(data, seq=8):
    x, y = [], []
    for i in range(len(data)-seq): x.append(data[i:i+seq]); y.append(data[i+seq, 0])
    return np.array(x), np.array(y)

print("🚀 Iniciando entrenamiento completo...")
for est in estaciones:
    tf.keras.backend.clear_session()
    subset = (df if est == 'GLOBAL' else df[df['Estacion'] == est])[features].fillna(0).values

    # Escalado Robusto
    scaler = StandardScaler().fit(subset)
    joblib.dump(scaler, os.path.join(RUTA_OUTPUT, f"scaler_{est.replace(' ', '_')}.save"))

    X, y = create_sequences(scaler.transform(subset))

    # Arquitectura LSTM completa
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(8, 16)),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.2), # Dropout para evitar overfitting
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=15, batch_size=32, verbose=1) # Épocas 15
    model.save(os.path.join(RUTA_OUTPUT, f"modelo_{est.replace(' ', '_')}.keras"))
    print(f"✅ Procesado: {est}")

# 5. Dashboard Predictivo Interactivo
cache = {}
dd_est = widgets.Dropdown(options=estaciones, description='Localidad:')
dd_con = widgets.Dropdown(options=features, description='Contaminante:')
dd_anio = widgets.Dropdown(options=[2026, 2027], description='Año:')
dd_mes = widgets.Dropdown(options=list(range(1, 13)), description='Mes:')
btn = widgets.Button(description='Predecir IA', button_style='danger')
out = widgets.Output()

def predecir(b):
    out.clear_output()
    est = dd_est.value.replace(' ', '_')
    with out:
        if est not in cache:
            cache[est] = {'m': tf.keras.models.load_model(os.path.join(RUTA_OUTPUT, f"modelo_{est}.keras")),
                         's': joblib.load(os.path.join(RUTA_OUTPUT, f"scaler_{est}.save"))}

        subset = df if dd_est.value == 'GLOBAL' else df[df['Estacion'] == dd_est.value]
        # Cálculo de promedio histórico mensual
        m_val = subset[subset['time'].dt.month == int(dd_mes.value)][features].mean().values.reshape(1, -1)
        # Reshape for sequence (1, 8, 16)
        m_seq = np.repeat(m_val, 8, axis=0).reshape(1, 8, 16)

        # Inferencia
        scale_input = cache[est]['s'].transform(m_seq.reshape(8, 16)).reshape(1, 8, 16)
        pred_scaled = cache[est]['m'].predict(scale_input, verbose=0)[0, 0]

        # Desescalado preciso
        dummy = np.zeros((1, 16))
        idx = features.index(dd_con.value)
        dummy[0, idx] = pred_scaled
        pred_base = cache[est]['s'].inverse_transform(dummy)[0, idx]

        # Corrección OLS
        slope_an = tasas_anuales.get(dd_est.value, 0)
        slope_mes = tasas_mensuales.get((dd_est.value, dd_mes.value), 0)
        final = max(0, pred_base + ((slope_an + slope_mes) * (dd_anio.value - 2024)))

        print(f"🔮 Predicción {dd_con.value} ({dd_mes.value}/{dd_anio.value}): {final:.2f} µg/m³")

btn.on_click(predecir)
display(dd_est, dd_con, dd_anio, dd_mes, btn, out)

🚀 Calculando tendencias OLS...
🚀 Iniciando entrenamiento completo...
Epoch 1/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - loss: 0.3593
Epoch 2/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.2629
Epoch 3/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 0.2471
Epoch 4/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.2353
Epoch 5/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.2314
Epoch 6/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.2190
Epoch 7/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.2100
Epoch 8/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.2039
Epoch 9/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.1996
Epoch 10/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.1949
Epoch 11/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.1834
Epoch 12/15
913/913 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - loss: 0.1810
Epoch 13/15
558/913 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.1619

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
import tensorflow as tf
from scipy import stats
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# 1. Configuración
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
PATH_BASE = "/content/drive/MyDrive/TFM_Profesorado/"
RUTA_OUTPUT = os.path.join(PATH_BASE, "Modelos_IA_Listos")
os.makedirs(RUTA_OUTPUT, exist_ok=True)

# 2. Carga
df = pd.read_parquet(f"{PATH_BASE}Murcia_Dataset_Completo_Global_3H.parquet")
df['time'] = pd.to_datetime(df['time'])
df['anio'] = df['time'].dt.year
features = ['PM10', 'PM2.5', 'NO2', 'Ozono', 'CO', 'SO2', 'Temp', 'Hum',
            'Viento_Vel', 'Viento_Dir', 'Presion', 'Nubes', 'UV_Real',
            'UV_Cielo_Despejado', 'Lluvia', 'Radiacion']

# 3. Tendencias OLS (Cálculo de la mejora anual real)
print("🚀 Calculando tendencias OLS...")
tasas_anuales, tasas_mensuales = {}, {}
estaciones = list(df['Estacion'].unique()) + ['GLOBAL']

for est in estaciones:
    sub = df if est == 'GLOBAL' else df[df['Estacion'] == est]
    m_anio = sub.groupby('anio')['NO2'].mean().reset_index()
    slope, _, _, _, _ = stats.linregress(m_anio['anio'], m_anio['NO2'])
    tasas_anuales[est] = slope
    for mes in range(1, 13):
        m = sub[sub['time'].dt.month == mes].groupby('anio')['NO2'].mean().reset_index()
        tasas_mensuales[(est, mes)] = stats.linregress(m['anio'], m['NO2'])[0] if len(m) > 2 else 0

# 4. Entrenamiento Masivo (StandardScaler por localidad)
print("🚀 Iniciando entrenamiento optimizado...")
def create_sequences(data, seq=8):
    x, y = [], []
    for i in range(len(data)-seq): x.append(data[i:i+seq]); y.append(data[i+seq, 0])
    return np.array(x), np.array(y)

for est in estaciones:
    tf.keras.backend.clear_session()
    subset = (df if est == 'GLOBAL' else df[df['Estacion'] == est])[features].fillna(0).values

    # Escalado Robusto por localidad
    scaler = StandardScaler().fit(subset)
    joblib.dump(scaler, os.path.join(RUTA_OUTPUT, f"scaler_{est.replace(' ', '_')}.save"))

    X, y = create_sequences(scaler.transform(subset))

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(8, 16)),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=5, batch_size=128, verbose=0)
    model.save(os.path.join(RUTA_OUTPUT, f"modelo_{est.replace(' ', '_')}.keras"))
    print(f"✅ Procesado: {est}")

# 5. Dashboard Predictivo Interactivo
cache = {}
dd_est = widgets.Dropdown(options=estaciones, description='Localidad:')
dd_con = widgets.Dropdown(options=features, description='Contaminante:')
dd_anio = widgets.Dropdown(options=[2026, 2027], description='Año:')
dd_mes = widgets.Dropdown(options=list(range(1, 13)), description='Mes:')
btn = widgets.Button(description='Ejecutar Predicción', button_style='danger')
out = widgets.Output()

def predecir(b):
    out.clear_output()
    est = dd_est.value.replace(' ', '_')
    with out:
        if est not in cache:
            cache[est] = {'m': tf.keras.models.load_model(os.path.join(RUTA_OUTPUT, f"modelo_{est}.keras")),
                         's': joblib.load(os.path.join(RUTA_OUTPUT, f"scaler_{est}.save"))}

        subset = df if dd_est.value == 'GLOBAL' else df[df['Estacion'] == dd_est.value]
        # Promedio mensual histórico, repetido 8 veces para cumplir secuencia (8, 16)
        m_val = subset[subset['time'].dt.month == dd_mes.value][features].mean().values.reshape(1, -1)
        m_seq = np.repeat(m_val, 8, axis=0).reshape(1, 8, 16)

        # Inferencia
        scale_input = cache[est]['s'].transform(m_seq.reshape(8, 16)).reshape(1, 8, 16)
        pred_scaled = cache[est]['m'].predict(scale_input, verbose=0)[0, 0]

        # Desescalado preciso (dummy para recrear el scaler de 16 cols)
        dummy = np.zeros((1, 16))
        idx = features.index(dd_con.value)
        dummy[0, idx] = pred_scaled
        pred_base = cache[est]['s'].inverse_transform(dummy)[0, idx]

        final = max(0, pred_base + ((tasas_anuales[dd_est.value] + tasas_mensuales[(dd_est.value, dd_mes.value)]) * (dd_anio.value - 2024)))
        print(f"🔮 Predicción {dd_con.value} ({dd_mes.value}/{dd_anio.value}): {final:.2f} µg/m³")

btn.on_click(predecir)
display(dd_est, dd_con, dd_anio, dd_mes, btn, out)

🚀 Calculando tendencias OLS...
🚀 Iniciando entrenamiento optimizado...
✅ Procesado: Alcantarilla
✅ Procesado: Archena
✅ Procesado: Caravaca de la Cruz
✅ Procesado: Cartagena
✅ Procesado: Cieza
✅ Procesado: Jumilla
✅ Procesado: Lorca
✅ Procesado: Mazarrón
✅ Procesado: Molina de Segura
✅ Procesado: Murcia Capital
✅ Procesado: San Javier
✅ Procesado: Torre-Pacheco
✅ Procesado: Totana
✅ Procesado: Yecla
✅ Procesado: Águilas
✅ Procesado: GLOBAL


Dropdown(description='Localidad:', options=('Alcantarilla', 'Archena', 'Caravaca de la Cruz', 'Cartagena', 'Ci…

Dropdown(description='Contaminante:', options=('PM10', 'PM2.5', 'NO2', 'Ozono', 'CO', 'SO2', 'Temp', 'Hum', 'V…

Dropdown(description='Año:', options=(2026, 2027), value=2026)

Dropdown(description='Mes:', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12), value=1)

Button(button_style='danger', description='Ejecutar Predicción', style=ButtonStyle())

Output()